So far we have ben working with synthetic data that arrived in ready-made tensors. However to apply deep learning in the wild we must extract messy data in the abitrary formats and preprocess it to suit our needs. Fortunately the *pandas* library can do much of the heavy lifting. This section while no substitue for a proper *pandas* tutorial will give you a crash crouse on some of the most common routines.


## 1. Reading the Dataset

Comma-separated values (CSV) files are ubiquitous for the storing of tabular (spreadsheet-
like) data.

In them, each line corresponds to one record and consists of several (comma-separated) fields, e.g., “Albert Einstein,March 14 1879,Ulm,Federal polytechnic school,field of gravitational physics”. To demonstrate how to load CSV files with pandas, we create a CSV file below `../data/house_tiny.csv`.

This file represents a dataset of homes, where each row corresponds to a distinct home and the columns correspond to the number of rooms (NumRooms), the roof type (RoofType), and the price (Price).

In [5]:
import os

os.makedirs(os.path.join('..','data'),exist_ok = True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')

# Opening file and adding data
with open(data_file, 'w') as f:
  f.write('''NumRooms,RoofType,Price
    NA,NA,127500
    2,NA,106000
    4,Slate,178100
    NA,NA,140000''')

Now let’s import pandas and load the dataset with read_csv.

In [7]:
import pandas as pd

data = pd.read_csv(data_file)
print(data)

  NumRooms RoofType   Price
0       NA      NaN  127500
1        2      NaN  106000
2        4    Slate  178100
3       NA      NaN  140000


## 2. Data Preperation

In supervised learning, we train models to predict a designated `target value`, given some
set of `input values`. Our first step in processing the dataset is to separate out columns cor-responding to input versus target values. We can select columns either by name or via integer-location based indexing (*iloc*).

You might have noticed that pandas replaced all CSV entries with value `NA` with a `special NaN` (not a number) value. This can also happen whenever an entry is empty, e.g.,“3„,270000”. These are called *missing values* and they are the **“bed bugs”** of data science,a persistent menace that you will confront throughout your career.

Depending upon the context, missing values might be handled either via *imputation* or *deletion*. *Imputation* re-places missing values with estimates of their values while deletion simply discards either those rows or those columns that contain missing values.

Here are some common imputation heuristics. For categorical input fields, we can treat `NaN`
as a category. Since the RoofType column takes values `Slate` and `NaN`, pandas can convert
this column into two columns `RoofType_Slate` and `RoofType_nan`. A row whose roof type
is Slate will set values of `RoofType_Slate` and `RoofType_nan` to `1` and `0`, respectively.
The converse holds for a row with a missing `RoofType value`.


In [10]:
inputs, targets = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms_    2  NumRooms_    4  NumRooms_    NA  NumRooms_nan  \
0           False           False             True         False   
1            True           False            False         False   
2           False            True            False         False   
3           False           False             True         False   

   RoofType_Slate  RoofType_nan  
0           False          True  
1           False          True  
2            True         False  
3           False          True  


For missing numerical values, one common heuristic is to replace the `NaN` entries with the
mean value of the corresponding column.

In [11]:
inputs = inputs.fillna(inputs.mean())
print(inputs)

   NumRooms_    2  NumRooms_    4  NumRooms_    NA  NumRooms_nan  \
0           False           False             True         False   
1            True           False            False         False   
2           False            True            False         False   
3           False           False             True         False   

   RoofType_Slate  RoofType_nan  
0           False          True  
1           False          True  
2            True         False  
3           False          True  


## 3. Conversion to the Tensor Format

Now all the entries in the inputs and targes are numerical we can load them into a tensor

In [13]:
import torch

x = torch.tensor(inputs.to_numpy(dtype = float))
y = torch.tensor(targets.to_numpy(dtype = float))

x, y

(tensor([[0., 0., 1., 0., 0., 1.],
         [1., 0., 0., 0., 0., 1.],
         [0., 1., 0., 0., 1., 0.],
         [0., 0., 1., 0., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

## 4. Discussion

You now know how to partition data columns, impute missing variables, and load pan-
das data into tensors. We will pick up some more data processing skills.While this crash course kept things simple, data processing can get hairy.

For example,rather than arriving in a single CSV file, our dataset might be spread across multiple files extracted from a relational database. For instance, in an e-commerce application, customer
addresses might live in one table and purchase data in another. Moreover, practitioners face
myriad data types beyond categorical and numeric, for example, text strings, images, audio
data, and point clouds. Oftentimes, advanced tools and efficient algorithms are required
in order to prevent data processing from becoming the biggest bottleneck in the machine
learning pipeline. These problems will arise when we get to computer vision and natural
language processing. Finally, we must pay attention to data quality.

Real-world datasets are often plagued by outliers, faulty measurements from sensors, and recording errors, which must be addressed before feeding the data into any model. Data visualization tools such as seaborn ,Bokeh, or matplotlib can help you to manually inspect the data and develop intuitions about the type of problems you may need to address.

# 5. Exercises

1. Try loading datasets, e.g., Abalone from the UCI Machine Learning Repository50 and inspect their properties. What fraction of them has missing values? What fraction of the variables is numerical, categorical, or text?

In [1]:
"""Install this library to access dataset"""
#pip install ucimlrepo

from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
default_of_credit_card_clients = fetch_ucirepo(id=350) 
  
# data (as pandas dataframes) 
X = default_of_credit_card_clients.data.features 
y = default_of_credit_card_clients.data.targets 
  
# metadata 
print(default_of_credit_card_clients.metadata) 
  
# variable information 
print(default_of_credit_card_clients.variables) 

{'uci_id': 350, 'name': 'Default of Credit Card Clients', 'repository_url': 'https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients', 'data_url': 'https://archive.ics.uci.edu/static/public/350/data.csv', 'abstract': "This research aimed at the case of customers' default payments in Taiwan and compares the predictive accuracy of probability of default among six data mining methods.", 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 30000, 'num_features': 23, 'feature_types': ['Integer', 'Real'], 'demographics': ['Sex', 'Education Level', 'Marital Status', 'Age'], 'target_col': ['Y'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Fri Mar 29 2024', 'dataset_doi': '10.24432/C55S3H', 'creators': ['I-Cheng Yeh'], 'intro_paper': {'ID': 365, 'type': 'NATIVE', 'title': 'The comparisons of data mining techniques for the predictive accuracy of 

In [5]:
X.shape

(30000, 23)

In [6]:
X.describe


<bound method NDFrame.describe of            X1  X2  X3  X4  X5  X6  X7  X8  X9  X10  ...     X14    X15    X16  \
0       20000   2   2   1  24   2   2  -1  -1   -2  ...     689      0      0   
1      120000   2   2   2  26  -1   2   0   0    0  ...    2682   3272   3455   
2       90000   2   2   2  34   0   0   0   0    0  ...   13559  14331  14948   
3       50000   2   2   1  37   0   0   0   0    0  ...   49291  28314  28959   
4       50000   1   2   1  57  -1   0  -1   0    0  ...   35835  20940  19146   
...       ...  ..  ..  ..  ..  ..  ..  ..  ..  ...  ...     ...    ...    ...   
29995  220000   1   3   1  39   0   0   0   0    0  ...  208365  88004  31237   
29996  150000   1   3   2  43  -1  -1  -1  -1    0  ...    3502   8979   5190   
29997   30000   1   2   2  37   4   3   2  -1    0  ...    2758  20878  20582   
29998   80000   1   3   1  41   1  -1   0   0    0  ...   76304  52774  11855   
29999   50000   1   2   1  46   0   0   0   0    0  ...   49764  36535  324

In [4]:
X.isna().value_counts()

X1     X2     X3     X4     X5     X6     X7     X8     X9     X10    X11    X12    X13    X14    X15    X16    X17    X18    X19    X20    X21    X22    X23  
False  False  False  False  False  False  False  False  False  False  False  False  False  False  False  False  False  False  False  False  False  False  False    30000
Name: count, dtype: int64

In [9]:
X.dtypes

X1     int64
X2     int64
X3     int64
X4     int64
X5     int64
X6     int64
X7     int64
X8     int64
X9     int64
X10    int64
X11    int64
X12    int64
X13    int64
X14    int64
X15    int64
X16    int64
X17    int64
X18    int64
X19    int64
X20    int64
X21    int64
X22    int64
X23    int64
dtype: object

2. Try indexing and selecting data columns by name rather than by column number. The
pandas documentation on indexing51 has further details on how to do this.

In [10]:
X.X1

0         20000
1        120000
2         90000
3         50000
4         50000
          ...  
29995    220000
29996    150000
29997     30000
29998     80000
29999     50000
Name: X1, Length: 30000, dtype: int64